# Чистовик: константный прогноз

Заготовка, которая уже проходит весь путь: читает `data/`, считает CV по фолдам, пишет `submission.csv` и `cv.json` и печатает `cv.json` последней строкой. Замените прогноз и разбиение на свои.

In [27]:
import json, sys
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from catboost import CatBoostRegressor

sys.path.insert(0, "data")
from metric_cv import quantile_quality, cv_json

tr = pd.read_csv("data/train.csv")
te = pd.read_csv("data/test.csv")
sample = pd.read_csv("data/sample_submission.csv")
y = tr["daily_orders"].to_numpy(float)
QUANTILES = (0.1, 0.5, 0.9)


X_tr = tr.drop(columns=['id', 'daily_orders'])
X_te = te.drop(columns=['id'])

cat_features = ['city', 'district_type', 'entrance_type']


def predict(X_tr, y_tr, X_val):
    """Константный прогноз: квантили спроса по всему train. Замените на модель."""
    model = CatBoostRegressor(cat_features=cat_features, verbose=500)
    model.fit(X_tr, y_tr)
    pred = model.predict(X_val)
    return np.c_[pred - 10, pred, pred + 10]
    # q = np.quantile(y_train, QUANTILES)
    # return np.tile(q, (n, 1))


In [28]:


# CV: четыре случайных фолда — замените разбиением, которое похоже на тест
splits = list(KFold(4, shuffle=True, random_state=0).split(tr))
oof = np.zeros((len(tr), 3))
for a, b in splits:
    oof[b] = predict(X_tr.iloc[a], y[a], X_tr.iloc[b])
folds = [100 * quantile_quality(y[b], oof[b, 0], oof[b, 1], oof[b, 2], tr["city"].iloc[b]) for _, b in splits]

# посылка: колонки и строки как в sample_submission.csv, 0 <= p10 <= p50 <= p90 <= 5000
test = np.round(predict(X_tr, y, X_te), 1)
sub = pd.DataFrame({"id": te["id"],
                    "predicted_daily_orders_p10": test[:, 0],
                    "predicted_daily_orders_p50": test[:, 1],
                    "predicted_daily_orders_p90": test[:, 2]})
assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample)
assert set(sub["id"]) == set(sample["id"]) and (sub.iloc[:, 1:].to_numpy() >= 0).all()
sub.to_csv("submission.csv", index=False)

cv = cv_json("catboost_const_10", [f / 100 for f in folds])   # пишет cv.json
print(json.dumps(cv))                                  # последняя строка — для проверки воспроизводимости


Learning rate set to 0.054268
0:	learn: 107.3920105	total: 11.9ms	remaining: 11.9s
500:	learn: 16.9100429	total: 4.16s	remaining: 4.15s
999:	learn: 12.4502971	total: 8.23s	remaining: 0us
Learning rate set to 0.054268
0:	learn: 113.9627028	total: 7.96ms	remaining: 7.96s
500:	learn: 17.7850846	total: 4.11s	remaining: 4.1s
999:	learn: 12.8374805	total: 8.26s	remaining: 0us
Learning rate set to 0.054268
0:	learn: 114.6269594	total: 7.89ms	remaining: 7.89s
500:	learn: 18.3457851	total: 4.21s	remaining: 4.19s
999:	learn: 12.9939379	total: 8.32s	remaining: 0us
Learning rate set to 0.054268
0:	learn: 108.4791398	total: 7.88ms	remaining: 7.87s
500:	learn: 17.6582610	total: 4.07s	remaining: 4.05s
999:	learn: 12.7975263	total: 8.27s	remaining: 0us
Learning rate set to 0.056791
0:	learn: 110.8747619	total: 7.66ms	remaining: 7.66s
500:	learn: 18.6031057	total: 4.37s	remaining: 4.35s
999:	learn: 13.9479342	total: 8.71s	remaining: 0us
{"name": "catboost_const_10", "metric": "quantile_quality_x100", "